# 01 - Get the raw data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/demand-forecasting-capstone/blob/main/notebooks/01_data_acquisition.ipynb)

**Goal:** download M5, Walmart and NOAA weather into `data/raw/` (never into git).

**You can skip this notebook** while using `synthetic` data. For the real data you need a free Kaggle account (see below).

How to run: `Runtime -> Run all` (Colab) or run the cells from top to bottom. The default dataset is `synthetic`, which needs no downloads.

In [ ]:
# ---- Setup: works in Google Colab and on your own computer. Nothing to upload. ----
REPO_URL = "https://github.com/YOUR-USERNAME/demand-forecasting-capstone.git"   # <- set once, the same in every notebook
BRANCH = "main"
USE_DRIVE = False   # True = use a copy of the project folder in your Google Drive instead of GitHub

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if not os.path.exists("src/demand_agents"):
    if IN_COLAB and USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.chdir("/content/drive/MyDrive/demand-forecasting-capstone")
    elif IN_COLAB:
        if not os.path.exists("/content/project"):
            subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/project"], check=True)
        os.chdir("/content/project")
    elif os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")                                   # running locally from the notebooks folder
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[colab]"], check=True)
sys.path.insert(0, os.path.abspath("src"))
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 120)
from IPython.display import display, Image, Markdown
print("Project folder:", os.getcwd())

## What gets downloaded

| Dataset | Source | Needs |
|---|---|---|
| **M5** (daily unit sales of Walmart items, 2011-2016) | Kaggle competition `m5-forecasting-accuracy` | Kaggle token **and** you must click *Join / Late Submission* on the competition page once |
| **Walmart** (weekly store sales, 2010-2012) | Kaggle dataset `yasserh/walmart-dataset` | Kaggle token |
| **NOAA weather** (daily temperature) | public GHCN-Daily files at ncei.noaa.gov | nothing |

Switch on only what you need.

In [ ]:
DOWNLOAD_M5 = False
DOWNLOAD_WALMART = False
DOWNLOAD_NOAA = False

## Kaggle credentials

On kaggle.com open *Settings -> API*. Click **Generate New Token** (a long text) or **Create Legacy API Key** (a `kaggle.json` file). Either works. Treat it like a password: never paste it into a cell that you save or share. The box below hides what you type.

In [ ]:
if DOWNLOAD_M5 or DOWNLOAD_WALMART:
    import getpass
    from demand_agents.data.acquire import kaggle_ready
    if not kaggle_ready():
        token = getpass.getpass("Paste your Kaggle API token (hidden). Press Enter to upload kaggle.json instead: ").strip()
        if token:
            os.environ["KAGGLE_API_TOKEN"] = token
        elif IN_COLAB:
            from google.colab import files
            import shutil
            files.upload()
            os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
            shutil.copy("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json")); os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"], check=True)
    print("Kaggle ready:", kaggle_ready())

In [ ]:
from demand_agents.data import acquire
if DOWNLOAD_M5:
    print("M5 ->", acquire.download_m5("data/raw/m5"))
if DOWNLOAD_WALMART:
    print("Walmart ->", acquire.download_walmart("data/raw/walmart"))
if DOWNLOAD_NOAA:
    print("NOAA ->", acquire.download_noaa("data/raw/noaa"))     # one file per state station

## What is in `data/raw/` now

In [ ]:
import pathlib
rows = [(str(p), round(p.stat().st_size / 1e6, 2)) for p in sorted(pathlib.Path("data/raw").rglob("*")) if p.is_file() and p.name != ".gitkeep"]
display(pd.DataFrame(rows, columns=["file", "MB"]) if rows else "nothing downloaded yet (fine for synthetic data)")

**Next:** `02_data_audit_and_cleaning.ipynb`.